# 02. MCP in one process

The Model Context Protocol is a standard way for an application to offer tools
and for a model client to discover and call them. Server and client normally sit
in different processes; here both live in this notebook and talk over an
in-memory transport, so you see the real protocol without any setup.

Most of this notebook is offline. Only the last cell, whose heading says
"requires LM Studio", lets the model pick the tools.

In [ ]:
%pip install -q openai==2.53.0 "mcp[cli]==2.0.0"

## 1. Constants

The same constant block as the other notebooks. This notebook only needs the
chat model, but the full set is kept so the notebooks stay interchangeable.

In [ ]:
from pathlib import Path

LM_STUDIO_BASE_URL = "http://127.0.0.1:1234/v1"
LM_STUDIO_API_KEY = "lm-studio"
CHAT_MODEL = "qwen/qwen3.5-9b"
EMBEDDING_MODEL = "text-embedding-qwen3-embedding-4b"
TOP_K = 3
DOCUMENTS_DIR = Path("documents")
RUN_LM_STUDIO_DEMO = True  # set to False to run only the offline cells

print("Chat model:", CHAT_MODEL)
print("Documents directory:", DOCUMENTS_DIR.resolve())
print("LM Studio cells enabled:", RUN_LM_STUDIO_DEMO)

## 2. Define an MCP server with three tools (offline)

A tool is a plain Python function plus a docstring. The decorator publishes the
name, the docstring and the argument types as a JSON schema, which is how a
model later learns what it may call. The factory returns a new server on every
call, so rerunning this cell never registers a tool twice.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

from mcp.server import MCPServer


def make_mcp_server() -> MCPServer:
    server = MCPServer("notebook-mcp-demo")

    @server.tool()
    def add(a: int, b: int) -> int:
        """Return the sum of two integers."""
        return a + b

    @server.tool()
    def current_time(timezone: str = "UTC") -> str:
        """Return the current wall-clock time in an IANA timezone."""
        try:
            tz = ZoneInfo(timezone)
        except ZoneInfoNotFoundError:
            return f"Unknown timezone: {timezone!r}"
        return datetime.now(tz).isoformat(timespec="seconds")

    @server.tool()
    def list_documents() -> list[str]:
        """List the Markdown files in the local documents directory."""
        return [
            path.relative_to(DOCUMENTS_DIR).as_posix()
            for path in sorted(DOCUMENTS_DIR.rglob("*.md"))
        ]

    return server


mcp_server = make_mcp_server()
print("MCP server created with tools: add, current_time, list_documents")

## 3. Discover the tools through a client session (offline)

This is not a Python function call. `ClientSession` performs the MCP handshake
over the in-memory transport and asks the server what it offers. Look at the
printed schema: that JSON is exactly what a model needs to call the tool.

In [ ]:
import json
from typing import Any

from mcp import ClientSession
from mcp.client._memory import InMemoryTransport

async with InMemoryTransport(mcp_server, raise_exceptions=True) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        advertised = await session.list_tools()
        for tool in advertised.tools:
            print(f"{tool.name}: {tool.description}")
        print("\nSchema of 'add':")
        print(json.dumps(advertised.tools[0].input_schema, indent=2))

## 4. Call the tools through the protocol (offline)

The client sends a call request with arguments and gets back content blocks.
Results always arrive as text blocks, so a small helper joins them. The output
shows the three results.

In [ ]:
def tool_result_text(result: Any) -> str:
    return "\n".join(getattr(block, "text", str(block)) for block in result.content).strip()


async with InMemoryTransport(mcp_server, raise_exceptions=True) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        sum_result = await session.call_tool("add", {"a": 17, "b": 25})
        time_result = await session.call_tool("current_time", {"timezone": "Europe/Bucharest"})
        documents_result = await session.call_tool("list_documents", {})
        print("add =>", tool_result_text(sum_result))
        print("current_time =>", tool_result_text(time_result))
        print("list_documents =>", tool_result_text(documents_result))

## 5. Let the model choose the tools (requires LM Studio)

The MCP schemas are translated into OpenAI function definitions and sent with
the question. The model answers with tool calls instead of text, the client runs
them through MCP and sends the results back, and the loop repeats until the model
replies with plain text. Watch the `tool:` lines to see what it decided to call.

In [ ]:
from openai import OpenAI


def openai_tool_schemas(mcp_tools: Any) -> list[dict]:
    return [
        {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description or "",
                "parameters": tool.input_schema or {"type": "object"},
            },
        }
        for tool in mcp_tools.tools
    ]


QUESTION = "What is 17 plus 25, and what time is it in Europe/Bucharest?"

if RUN_LM_STUDIO_DEMO:
    llm = OpenAI(base_url=LM_STUDIO_BASE_URL, api_key=LM_STUDIO_API_KEY)
    agent_server = make_mcp_server()
    async with InMemoryTransport(agent_server, raise_exceptions=True) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            schemas = openai_tool_schemas(await session.list_tools())
            messages = [
                {"role": "system", "content": "Use the MCP tools when helpful; answer concisely."},
                {"role": "user", "content": QUESTION},
            ]
            for step in range(5):
                response = llm.chat.completions.create(
                    model=CHAT_MODEL,
                    messages=messages,
                    tools=schemas,
                    tool_choice="auto",
                    temperature=0.2,
                )
                message = response.choices[0].message
                calls = message.tool_calls or []
                if not calls:
                    print(message.content or "")
                    break
                messages.append({
                    "role": "assistant",
                    "content": message.content or "",
                    "tool_calls": [call.model_dump() for call in calls],
                })
                for call in calls:
                    arguments = json.loads(call.function.arguments or "{}")
                    result = await session.call_tool(call.function.name, arguments)
                    text = tool_result_text(result)
                    print(f"tool: {call.function.name}({arguments}) -> {text}")
                    messages.append({"role": "tool", "tool_call_id": call.id, "content": text})
            else:
                print("Stopped after five tool-selection steps.")
else:
    print("Skipped the tool loop. Set RUN_LM_STUDIO_DEMO = True to enable it.")